In [ ]:
from ultralytics import YOLO
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import scipy.ndimage as ndimage
from scipy.ndimage import gaussian_filter1d
import torchvision
from PIL import Image
import ttach as tta

In [ ]:
def get_prediction(p):
    l=[]
    for box in p[0].boxes:
        l.append([float(box.xyxy[0][0]) , float(box.xyxy[0][1]) , float(box.xyxy[0][2]) , float(box.xyxy[0][3]) , float(box.conf[0]),int(box.cls[0])])
    #l=np.array(l)
    l=torch.tensor(l)
    return l



def auc(arr):
    '''Returns normalized Area Under Curve of the array.'''
    return (arr.sum() - arr[0] / 2 - arr[-1] / 2) / (arr.shape[0] - 1)  # auc formula



def causal_metric(model, img, bbox, saliency_map, mode, step, kernel_width=0.25):
    '''
    model: type(nn.Module)
    img: type(np.ndarray) - shape:[H, W, 3]
    bbox: type(tensor) - shape:[num_boxes, (4 + 1 + num_classes + 1)] - Predicted bboxes
    saliency_map: type(np.ndarray) - shape:[num_boxes, H, W]
    mode: type(str) - Select deletion or insertion metric ('del' or 'ins')
    step: number of pixels modified per one iteration
    Return: deletion/insertion metric and number of objects.
    '''

    del_ins = np.zeros(2)
    count = np.zeros(2)
    HW = saliency_map.shape[1] * saliency_map.shape[2]
    n_steps = (HW + step - 1) // step
    l_image=[]
    l_scores=[]
    print(n_steps)
    for idx in range(saliency_map.shape[0]):
        target_cls = bbox[idx][-1]
        if mode == 'del':
            start = img.copy()
            finish = np.zeros_like(start)
        else:
            start = cv2.GaussianBlur(img, (51, 51), 0)
            finish = img.copy()
        start=cv2.cvtColor(start,cv2.COLOR_BGR2RGB)
        finish=cv2.cvtColor(finish,cv2.COLOR_BGR2RGB)
        salient_order = np.flip(np.argsort(saliency_map[idx].reshape(HW, -1), axis=0), axis=0)
        y = salient_order // img.shape[1]
        x = salient_order - y*img.shape[1]
        scores = np.zeros(n_steps + 1)
        with torch.no_grad():
            for i in range(n_steps + 1):
                if i%100==0:
                  print(int((i/n_steps)*100))
                temp_ious = []
                temp_score = []
                torch_start = torch.from_numpy(start.transpose(2, 0, 1)).unsqueeze(0).float()
                out = model((torch_start/255).cuda(),verbose=False,conf=0.25)
                p_box = get_prediction(out)

                if len(p_box) == 0:
                    scores[i] = 0
                else:
                    for box in p_box:
                        sample_cls = box[-1]
                        sample_box = box[:4]
                        sample_score = box[4]
                        iou = torchvision.ops.box_iou(sample_box[:4].unsqueeze(0), bbox[idx][:4].unsqueeze(0)).cpu().item()
                        weights = sample_score
                        if target_cls != sample_cls :
                            iou = 0
                            weights = torch.tensor(0.)
                        temp_ious.append(iou)
                        s_score = weights*iou
                        temp_score.append(s_score)
                    max_score = temp_score[np.argmax(temp_ious)]
                    scores[i] = max_score
                """
                if i in range(0,1000):
                    c=start.copy()
                    l_image.append(c)
                    l_scores.append(scores[i])
                """
                x_coords = x[step * i:step * (i+1), :]
                y_coords = y[step * i:step * (i+1), :]
                start[y_coords, x_coords, :] = finish[y_coords, x_coords, :]

        del_ins[int(target_cls)] += auc(scores)
        count[int(target_cls)] += 1
    return del_ins, count , scores


In [ ]:
bbox=[        x1,y1,x2,y2   , 0.88        ,  1]
img=cv2.imread(file_path)

confidence=bbox[4]
bbox=torch.tensor(bbox).reshape(1,6)
l=[normalized_final_heatmap_smoothed,naive_heatmap,gradcam_heatmap]
#l1=[normalized_final_heatmap_smoothed,normalized_final_heatmap_non_smoothed,naive_heatmap]
saliency_map=l[0].copy()
saliency_map=saliency_map.reshape(1,saliency_map.shape[0],saliency_map.shape[1])
mode1="ins"
mode2="del"
step=50
del_ins1,count1,scores1=causal_metric(model, img, bbox, saliency_map, mode1, step)
del_ins2,count2,scores2=causal_metric(model, img, bbox, saliency_map, mode2, step)